# 🎨 DiffuSVG: Complete Experimental Pipeline

**Prompt → Diffusion → Image → VLM → SVG**

## What this notebook does:
1. **Phase 1**: Generate vector-style images with SDXL
2. **Phase 2**: Use VLM (Claude/GPT-4V) to convert images to SVG
3. **Phase 3**: Iterative refinement with visual feedback
4. **Evaluation**: Compute CLIP Score, DINO Score, validity metrics

**Runtime**: ~2-4 hours on T4 GPU

⚠️ **IMPORTANT**: Go to `Runtime > Change runtime type > T4 GPU`

In [ ]:
#@title 🔑 **Configuration** { display-mode: "form" }

API_PROVIDER = "anthropic"  #@param ["anthropic", "openai"]
API_KEY = ""  #@param {type:"string"}
NUM_TEST_PROMPTS = 50  #@param {type:"slider", min:10, max:200, step:10}
MAX_REFINEMENT_ITERATIONS = 3  #@param {type:"slider", min:0, max:5, step:1}

import os
if API_PROVIDER == "anthropic":
    os.environ["ANTHROPIC_API_KEY"] = API_KEY
else:
    os.environ["OPENAI_API_KEY"] = API_KEY

print(f"✅ Configured for {API_PROVIDER.upper()}")
print(f"📊 Will evaluate on {NUM_TEST_PROMPTS} prompts")

In [ ]:
#@title 📦 **Install Dependencies** (~3 min)
%%capture
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q "huggingface_hub>=0.16.0"
!pip install -q diffusers[torch]>=0.30.0 transformers accelerate peft
!pip install -q anthropic openai
!pip install -q cairosvg pillow numpy pandas matplotlib seaborn tqdm
!pip install -q open_clip_torch
!apt-get install -qq libcairo2-dev
print("✅ Dependencies installed!")

In [ ]:
#@title 🔍 **Verify GPU**
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("❌ No GPU! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
#@title 📚 **Import Libraries**
import os, io, re, json, base64, random, warnings, time
from pathlib import Path
from typing import Optional, List, Dict, Any
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cairosvg

import torch
import torch.nn.functional as F
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
import open_clip

warnings.filterwarnings('ignore')
print("✅ Imports successful!")

In [ ]:
#@title 🛠️ **SVG Utilities**

class SVGUtils:
    @staticmethod
    def extract_svg(text: str) -> Optional[str]:
        patterns = [r'<svg[\s\S]*?</svg>', r'```(?:svg|xml)?\s*(<svg[\s\S]*?</svg>)\s*```']
        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                svg = match.group(0) if '<svg' in match.group(0) else match.group(1)
                if '<svg' in svg.lower(): return svg.strip()
        if text.strip().lower().startswith('<svg'):
            end = text.lower().find('</svg>')
            if end != -1: return text[:end + 6].strip()
        return None
    
    @staticmethod
    def validate_svg(svg_code: str) -> bool:
        try:
            if not svg_code or '<svg' not in svg_code.lower(): return False
            cairosvg.svg2png(bytestring=svg_code.encode('utf-8'), output_width=64, output_height=64)
            return True
        except: return False
    
    @staticmethod
    def repair_svg(svg_code: str) -> str:
        if not svg_code: return '<svg viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg"></svg>'
        if 'xmlns' not in svg_code: svg_code = svg_code.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
        if 'viewBox' not in svg_code.lower(): svg_code = svg_code.replace('<svg', '<svg viewBox="0 0 200 200"', 1)
        return svg_code
    
    @staticmethod
    def render_svg(svg_code: str, size: int = 512) -> Optional[Image.Image]:
        try:
            png_data = cairosvg.svg2png(bytestring=svg_code.encode('utf-8'), output_width=size, output_height=size)
            return Image.open(io.BytesIO(png_data)).convert('RGB')
        except: return None
    
    @staticmethod
    def count_elements(svg_code: str) -> Dict[str, int]:
        elements = ['path', 'rect', 'circle', 'ellipse', 'polygon', 'line']
        counts = {e: len(re.findall(f'<{e}[\\s/>]', svg_code, re.IGNORECASE)) for e in elements}
        counts['total'] = sum(counts.values())
        return counts

print("✅ SVGUtils loaded")

In [ ]:
#@title 📊 **Metrics Calculator**

class MetricsCalculator:
    def __init__(self, device='cuda'):
        self.device = device
        print("Loading CLIP...")
        self.clip_model, _, self.clip_preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
        self.clip_model = self.clip_model.to(device).eval()
        self.tokenizer = open_clip.get_tokenizer('ViT-B-32')
        print("Loading DINO...")
        self.dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', verbose=False).to(device).eval()
        print("✅ Metrics models loaded")
    
    @torch.no_grad()
    def clip_score(self, image: Image.Image, text: str) -> float:
        img_t = self.clip_preprocess(image).unsqueeze(0).to(self.device)
        txt_t = self.tokenizer([text]).to(self.device)
        img_f = F.normalize(self.clip_model.encode_image(img_t), dim=-1)
        txt_f = F.normalize(self.clip_model.encode_text(txt_t), dim=-1)
        return (img_f @ txt_f.T).item() * 100
    
    @torch.no_grad()
    def dino_similarity(self, img1: Image.Image, img2: Image.Image) -> float:
        def prep(img):
            img = img.resize((224, 224))
            arr = np.array(img).astype(np.float32) / 255.0
            arr = (arr - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
            return torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).float().to(self.device)
        f1, f2 = self.dino_model(prep(img1)), self.dino_model(prep(img2))
        return max(0, F.cosine_similarity(f1, f2).item())
    
    def compute_all(self, svg_code: str, ref_img: Image.Image, prompt: str) -> Dict[str, Any]:
        result = {'valid': False, 'clip_score': 0.0, 'dino_score': 0.0, 'element_count': 0, 'path_count': 0}
        if not SVGUtils.validate_svg(svg_code): return result
        rendered = SVGUtils.render_svg(svg_code)
        if rendered is None: return result
        counts = SVGUtils.count_elements(svg_code)
        return {'valid': True, 'clip_score': self.clip_score(rendered, prompt), 'dino_score': self.dino_similarity(ref_img, rendered), 'element_count': counts['total'], 'path_count': counts['path']}

metrics_calc = MetricsCalculator()

In [ ]:
#@title 🎨 **Load Diffusion Model** (~2 min)

class DiffusionGenerator:
    STYLE_PREFIX = "flat vector illustration, clean edges, solid colors, simple shapes, "
    NEGATIVE = "gradient, photorealistic, blurry, complex textures, shadows, 3d, realistic"
    
    def __init__(self):
        print("Loading SDXL (takes ~2 min)...")
        self.pipe = StableDiffusionXLPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16,
            use_safetensors=True, variant="fp16"
        )
        self.pipe.scheduler = DPMSolverMultistepScheduler.from_config(self.pipe.scheduler.config)
        self.pipe = self.pipe.to("cuda")
        try: self.pipe.enable_xformers_memory_efficient_attention()
        except: pass
        print("✅ SDXL loaded")
    
    @torch.inference_mode()
    def generate(self, prompt: str, seed: int = None) -> Image.Image:
        gen = torch.Generator("cuda").manual_seed(seed) if seed else None
        return self.pipe(self.STYLE_PREFIX + prompt, negative_prompt=self.NEGATIVE,
                        num_inference_steps=25, guidance_scale=7.5, generator=gen).images[0]

diffusion = DiffusionGenerator()

In [ ]:
#@title 🤖 **VLM Code Generator**

class VLMGenerator:
    SYSTEM = """You are an SVG expert. Convert images to clean SVG code.
Rules: viewBox="0 0 200 200", xmlns required, use rect/circle/ellipse/polygon/path, solid hex colors, order by z-index.
Output ONLY valid SVG code starting with <svg and ending with </svg>."""
    
    def __init__(self, provider='anthropic'):
        self.provider = provider
        if provider == 'anthropic':
            import anthropic
            self.client = anthropic.Anthropic()
            self.model = 'claude-sonnet-4-20250514'
        else:
            import openai
            self.client = openai.OpenAI()
            self.model = 'gpt-4o'
        print(f"✅ VLM: {provider} ({self.model})")
    
    def _to_b64(self, img): 
        buf = io.BytesIO(); img.save(buf, format='PNG'); return base64.b64encode(buf.getvalue()).decode()
    
    def generate_svg(self, image: Image.Image, prompt: str) -> str:
        b64 = self._to_b64(image)
        if self.provider == 'anthropic':
            r = self.client.messages.create(model=self.model, max_tokens=4096, system=self.SYSTEM,
                messages=[{'role': 'user', 'content': [{'type': 'text', 'text': f'Convert to SVG: {prompt}'},
                    {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': b64}}]}])
            return r.content[0].text
        else:
            r = self.client.chat.completions.create(model=self.model, max_tokens=4096,
                messages=[{'role': 'system', 'content': self.SYSTEM},
                    {'role': 'user', 'content': [{'type': 'text', 'text': f'Convert to SVG: {prompt}'},
                        {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{b64}'}}]}])
            return r.choices[0].message.content
    
    def refine_svg(self, orig: Image.Image, rendered: Image.Image, svg: str) -> str:
        b64_o, b64_r = self._to_b64(orig), self._to_b64(rendered)
        prompt = f"Fix this SVG to match original better. Current SVG:\n{svg}\n\nOutput only corrected SVG."
        if self.provider == 'anthropic':
            r = self.client.messages.create(model=self.model, max_tokens=4096,
                messages=[{'role': 'user', 'content': [
                    {'type': 'text', 'text': 'Original:'}, {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': b64_o}},
                    {'type': 'text', 'text': 'Current render:'}, {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': b64_r}},
                    {'type': 'text', 'text': prompt}]}])
            return r.content[0].text
        else:
            r = self.client.chat.completions.create(model=self.model, max_tokens=4096,
                messages=[{'role': 'user', 'content': [
                    {'type': 'text', 'text': 'Original:'}, {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{b64_o}'}},
                    {'type': 'text', 'text': 'Current render:'}, {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{b64_r}'}},
                    {'type': 'text', 'text': prompt}]}])
            return r.choices[0].message.content

vlm = VLMGenerator(API_PROVIDER)

In [ ]:
#@title 📝 **Test Prompts**
TEST_PROMPTS = [
    "a red apple", "a yellow sun", "a blue circle", "a green tree", "a red heart",
    "a yellow star", "an orange carrot", "a pink flower", "a house with red roof",
    "a car with wheels", "a snowman with hat", "a rocket in space", "a cat face",
    "a wifi symbol", "a battery icon", "a music note", "a play button", "a settings gear",
    "a home icon", "a mail envelope", "a phone icon", "a camera icon", "a lock icon",
    "a mountain landscape", "a rainbow", "clouds in sky", "a crescent moon",
    "a slice of pizza", "a cup of coffee", "an ice cream cone", "a birthday cake",
    "a hamburger", "a donut", "a watermelon slice", "a banana", "a strawberry",
    "a hot air balloon", "a treasure chest", "a lighthouse", "a bicycle", "a guitar",
    "concentric circles", "a spiral", "overlapping squares", "a yin yang symbol",
    "a peace sign", "a target with rings", "a smiley face", "a thumbs up", "a lightning bolt"
]
random.seed(42)
selected_prompts = random.sample(TEST_PROMPTS, min(NUM_TEST_PROMPTS, len(TEST_PROMPTS)))
print(f"✅ Selected {len(selected_prompts)} prompts")
print(f"Samples: {selected_prompts[:5]}")

In [ ]:
#@title 🚀 **Pipeline Function**

def run_pipeline(prompt: str, seed: int = None, max_refine: int = 3, dino_thresh: float = 0.85) -> Dict:
    result = {'prompt': prompt, 'seed': seed, 'diff_img': None, 'svg': None, 'render': None,
              'metrics': {}, 'refine_iters': 0, 'success': False, 'error': None, 'time': {}}
    try:
        t0 = time.time()
        result['diff_img'] = diffusion.generate(prompt, seed)
        result['time']['diffusion'] = time.time() - t0
        
        t0 = time.time()
        raw = vlm.generate_svg(result['diff_img'], prompt)
        svg = SVGUtils.extract_svg(raw) or SVGUtils.repair_svg(raw)
        result['time']['vlm'] = time.time() - t0
        
        t0 = time.time()
        for i in range(max_refine):
            if not SVGUtils.validate_svg(svg): svg = SVGUtils.repair_svg(svg)
            if not SVGUtils.validate_svg(svg): break
            rendered = SVGUtils.render_svg(svg)
            if not rendered: break
            if metrics_calc.dino_similarity(result['diff_img'], rendered) >= dino_thresh: break
            ref_raw = vlm.refine_svg(result['diff_img'], rendered, svg)
            ref_svg = SVGUtils.extract_svg(ref_raw)
            if ref_svg and SVGUtils.validate_svg(ref_svg): svg = ref_svg
            result['refine_iters'] = i + 1
        result['time']['refine'] = time.time() - t0
        
        result['svg'] = svg
        if SVGUtils.validate_svg(svg):
            result['render'] = SVGUtils.render_svg(svg)
            result['metrics'] = metrics_calc.compute_all(svg, result['diff_img'], prompt)
            result['success'] = result['metrics']['valid']
    except Exception as e: result['error'] = str(e)
    return result

print("✅ Pipeline ready")

In [ ]:
#@title 🧪 **Test Single Example**
test = run_pipeline("a red apple", seed=42, max_refine=MAX_REFINEMENT_ITERATIONS)
print(f"Prompt: {test['prompt']}")
print(f"Success: {test['success']}, Refinements: {test['refine_iters']}")
print(f"Metrics: CLIP={test['metrics'].get('clip_score',0):.2f}, DINO={test['metrics'].get('dino_score',0):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(test['diff_img']); ax[0].set_title('Diffusion'); ax[0].axis('off')
if test['render']: ax[1].imshow(test['render']); ax[1].set_title(f"SVG (CLIP:{test['metrics']['clip_score']:.1f})"); ax[1].axis('off')
plt.show()
print(f"\nSVG Preview:\n{test['svg'][:300]}...")

In [ ]:
#@title 📊 **Run Full Evaluation** (~1-2 hours)
all_results = []
print(f"Evaluating {len(selected_prompts)} prompts (est. {len(selected_prompts)*2:.0f} min)...\n")

for i, prompt in enumerate(tqdm(selected_prompts)):
    result = run_pipeline(prompt, seed=42+i, max_refine=MAX_REFINEMENT_ITERATIONS)
    all_results.append(result)
    if (i+1) % 10 == 0:
        valid = sum(1 for r in all_results if r['success'])
        clip = np.mean([r['metrics']['clip_score'] for r in all_results if r['success']] or [0])
        print(f"  {i+1}/{len(selected_prompts)} | Valid: {valid} | Avg CLIP: {clip:.2f}")

print("\n✅ Evaluation complete!")

In [ ]:
#@title 📈 **Final Results**
valid = [r for r in all_results if r['success']]
summary = {
    'Total': len(all_results), 'Valid': len(valid),
    'Validity %': len(valid)/len(all_results)*100,
    'CLIP Mean': np.mean([r['metrics']['clip_score'] for r in valid]),
    'CLIP Std': np.std([r['metrics']['clip_score'] for r in valid]),
    'DINO Mean': np.mean([r['metrics']['dino_score'] for r in valid]),
    'DINO Std': np.std([r['metrics']['dino_score'] for r in valid]),
    'Avg Elements': np.mean([r['metrics']['element_count'] for r in valid]),
    'Avg Time (s)': np.mean([sum(r['time'].values()) for r in all_results]),
}

print("="*50)
print("       DIFFUSVG EXPERIMENTAL RESULTS")
print("="*50)
for k, v in summary.items():
    print(f"  {k:.<30} {v:.2f}" if isinstance(v, float) else f"  {k:.<30} {v}")
print("="*50)

In [ ]:
#@title 📊 **Visualizations**
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

clips = [r['metrics']['clip_score'] for r in valid]
axes[0,0].hist(clips, bins=15, color='steelblue', edgecolor='black')
axes[0,0].axvline(np.mean(clips), color='red', linestyle='--', label=f'Mean: {np.mean(clips):.2f}')
axes[0,0].set_xlabel('CLIP Score'); axes[0,0].set_title('CLIP Distribution'); axes[0,0].legend()

dinos = [r['metrics']['dino_score'] for r in valid]
axes[0,1].hist(dinos, bins=15, color='seagreen', edgecolor='black')
axes[0,1].axvline(np.mean(dinos), color='red', linestyle='--', label=f'Mean: {np.mean(dinos):.3f}')
axes[0,1].set_xlabel('DINO Score'); axes[0,1].set_title('DINO Distribution'); axes[0,1].legend()

axes[1,0].scatter(clips, dinos, alpha=0.6, color='purple')
axes[1,0].set_xlabel('CLIP'); axes[1,0].set_ylabel('DINO'); axes[1,0].set_title('CLIP vs DINO')

times = {'Diffusion': np.mean([r['time'].get('diffusion',0) for r in all_results]),
         'VLM': np.mean([r['time'].get('vlm',0) for r in all_results]),
         'Refine': np.mean([r['time'].get('refine',0) for r in all_results])}
axes[1,1].pie(times.values(), labels=times.keys(), autopct='%1.1f%%')
axes[1,1].set_title('Time Breakdown')

plt.tight_layout(); plt.savefig('results.png', dpi=150); plt.show()

In [ ]:
#@title 🖼️ **Sample Gallery**
sorted_r = sorted(valid, key=lambda x: x['metrics']['clip_score'], reverse=True)
top5, bot5 = sorted_r[:5], sorted_r[-5:]

fig, axes = plt.subplots(4, 5, figsize=(18, 14))
for i, r in enumerate(top5):
    axes[0,i].imshow(r['diff_img']); axes[0,i].set_title(r['prompt'][:18], fontsize=8); axes[0,i].axis('off')
    if r['render']: axes[1,i].imshow(r['render'])
    axes[1,i].set_title(f"CLIP:{r['metrics']['clip_score']:.1f}", fontsize=8); axes[1,i].axis('off')
for i, r in enumerate(bot5):
    axes[2,i].imshow(r['diff_img']); axes[2,i].set_title(r['prompt'][:18], fontsize=8); axes[2,i].axis('off')
    if r['render']: axes[3,i].imshow(r['render'])
    axes[3,i].set_title(f"CLIP:{r['metrics']['clip_score']:.1f}", fontsize=8); axes[3,i].axis('off')

fig.text(0.02, 0.78, 'TOP 5', fontweight='bold'); fig.text(0.02, 0.28, 'BOTTOM 5', fontweight='bold')
plt.tight_layout(); plt.savefig('gallery.png', dpi=150); plt.show()

In [ ]:
#@title 📋 **LaTeX Table**
latex = f"""\\begin{{table}}[h]
\\centering
\\caption{{DiffuSVG Results (N={len(all_results)})}}
\\begin{{tabular}}{{lc}}
\\toprule
\\textbf{{Metric}} & \\textbf{{Value}} \\\\
\\midrule
CLIP Score & {summary['CLIP Mean']:.2f} $\\pm$ {summary['CLIP Std']:.2f} \\\\
DINO Score & {summary['DINO Mean']:.3f} $\\pm$ {summary['DINO Std']:.3f} \\\\
Validity & {summary['Validity %']:.1f}\\% \\\\
Avg Elements & {summary['Avg Elements']:.1f} \\\\
Inference Time & {summary['Avg Time (s)']:.2f}s \\\\
\\bottomrule
\\end{{tabular}}
\\end{{table}}"""
print(latex)
with open('results.tex', 'w') as f: f.write(latex)

In [ ]:
#@title 💾 **Save & Download**
export = [{'prompt': r['prompt'], 'clip': r['metrics'].get('clip_score',0), 'dino': r['metrics'].get('dino_score',0),
           'valid': r['success'], 'elements': r['metrics'].get('element_count',0), 'time': sum(r['time'].values()),
           'svg': r['svg']} for r in all_results]

with open('results.json', 'w') as f: json.dump({'summary': summary, 'results': export}, f, indent=2)
pd.DataFrame(export).to_csv('results.csv', index=False)

!zip -q results.zip results.json results.csv results.png gallery.png results.tex

from google.colab import files
files.download('results.zip')
print("✅ Download started!")